# 03. Batch 2 model comparisons

Compare the frozen 72-predictor models with a **single half-month composite** (9 predictors) and **eight composites with spectral bands only** (40 predictors). Work on the fixed and green-up-aligned matched cohorts. A half-month median composite may contain multiple Sentinel-2 acquisitions.

The bin search uses five spatial folds inside development training. Rank the eight candidate bins by pooled out-of-fold parcel macro-F1, breaking ties in favor of the earlier bin. Score locked validation after selection. Both external comparisons use the same parcels as their frozen eight-bin counterparts. The aligned recipe uses a time series to locate green-up.

Use local frozen run artifacts from trusted sources. Set a new Batch 2 output directory to fit a fresh set of comparators. A completed comparator receipt allows the notebook to reuse the fitted packages.


In [ ]:
from pathlib import Path
import sys

root = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (path / "pyproject.toml").is_file())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from crop_pipeline.training import model_comparisons

training_run = root / "pipeline_outputs/training/run_fa40f5d6bb31"
testing_run = root / "pipeline_outputs/testing/run_81a0d0a2812e"
batch2_output = root / "pipeline_outputs/testing/run_81a0d0a2812e_batch2"


## Select on development and freeze

The cell fits comparators using the same training pixels, label weights, and forest settings as the frozen matched models. It records candidate scores, held-out parcel IDs, the chosen interval, predictor order, locked-validation results, and model hashes.


In [ ]:
comparators = batch2_output / "comparators.json"
if not comparators.is_file():
    comparators = model_comparisons.train_comparators(training_run, batch2_output)
print("Frozen comparator receipt:", comparators)


## Evaluate on the shared external parcels

Use JECAM first, California carrot, Denmark cabbage, non-JECAM potato, and the revised combined pool. This external benchmark has already been examined. The comparison leaves class F1 blank for absent reference crops and three-class macro-F1 blank for incomplete crop coverage.


In [ ]:
from IPython.display import Markdown, display

comparison = model_comparisons.score_comparators(comparators, testing_run)
report = model_comparisons.render_report(comparators, comparison, training_run)
display(Markdown(report.read_text(encoding="utf-8")))
print("Comparison tables and predictions:", batch2_output)


## Three-way external feature comparison

Compare interval-only (72), temporal-shape-only (48), and interval-plus-shape (120) on the same matched parcels. Reuse the two archived interval-plus-shape forests and fit only the two missing shape-only forests using frozen development features. Score external parcels after freezing.


In [ ]:
shape_receipt = batch2_output / "shape_comparison" / "comparators.json"
if not shape_receipt.is_file():
    shape_receipt = model_comparisons.freeze_shape_comparison(training_run, comparators)
shape_scores = model_comparisons.score_comparators(shape_receipt, testing_run)
shape_report = model_comparisons.render_shape_report(shape_receipt, shape_scores)
display(Markdown(shape_report.read_text(encoding="utf-8")))
print("Three-way comparison tables and predictions:", shape_receipt.parent)


## Indices-only follow-up

Keep all eight bins and use NDVI, NDRE, NDMI and SAVI for 32 predictors. Compare with the frozen bands-only and bands-plus-indices models on the same matched development and external parcels. The Random Forest settings and training weights are held fixed.


In [ ]:
from crop_pipeline.training import model_followups

indices_receipt = batch2_output / "indices_only_comparison" / "comparators.json"
if not indices_receipt.is_file():
    indices_receipt = model_followups.freeze_followup(training_run, comparators, "indices_only")
indices_scores = model_comparisons.score_comparators(indices_receipt, testing_run)
indices_report = model_followups.render_followup(indices_receipt, indices_scores)
display(Markdown(indices_report.read_text(encoding="utf-8")))
print("Indices comparison:", indices_receipt.parent)
